# 03 - Customer Segmentation (RFM + K-Means)
## Identify Customer Segments for Targeted Actions

This notebook segments customers using RFM (Recency, Frequency, Monetary) analysis
and K-Means clustering.

### Segments
- **VIP**: High value, frequent, recent buyers
- **Loyal**: Regular customers with consistent purchases
- **At Risk**: Previously active customers showing decline
- **Lost**: Inactive customers with no recent activity

### Output
- `trained_models/segmenter.joblib`
- `trained_models/segmenter_scaler.joblib`
- `trained_models/segment_mapping.joblib`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sys.path.insert(0, str(Path('.').resolve().parent))
from data.mapping import BRL_TO_DZD

sns.set_style('whitegrid')
MODEL_DIR = Path('../trained_models')
MODEL_DIR.mkdir(exist_ok=True)

print('Libraries loaded!')

## 1. Load Data

In [ ]:
DATA_DIR = Path('../data/olist')

orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')

# Merge
df = orders.merge(customers, on='customer_id', how='left')
pay_agg = payments.groupby('order_id').agg(payment_value=('payment_value', 'sum')).reset_index()
df = df.merge(pay_agg, on='order_id', how='left')

# Process
df['order_date'] = pd.to_datetime(df['order_purchase_timestamp'])
df['is_delivered'] = (df['order_status'] == 'delivered').astype(int)
df['payment_dzd'] = (df['payment_value'].fillna(0) * BRL_TO_DZD).round(2)

print(f'Total orders: {len(df)}')
print(f'Unique customers: {df["customer_unique_id"].nunique()}')

## 2. Compute RFM Features

In [ ]:
reference_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_unique_id').agg(
    recency=('order_date', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('payment_dzd', 'sum'),
    avg_order_value=('payment_dzd', 'mean'),
    success_rate=('is_delivered', 'mean'),
).reset_index()

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary', 'avg_order_value', 'success_rate']

print('RFM Statistics:')
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

# Visualize RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rfm['recency'].clip(upper=500).hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Recency (days since last order)')

rfm['frequency'].clip(upper=10).hist(bins=10, ax=axes[1], color='orange')
axes[1].set_title('Frequency (number of orders)')

rfm['monetary'].clip(upper=50000).hist(bins=50, ax=axes[2], color='green')
axes[2].set_title('Monetary (total spent in DZD)')

plt.tight_layout()
plt.show()

## 3. Find Optimal Number of Clusters (Elbow Method)

In [ ]:
# Standardize features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

# Elbow method
inertias = []
silhouettes = []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, km.labels_, sample_size=10000))
    print(f'K={k}: inertia={km.inertia_:.0f}, silhouette={silhouettes[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

## 4. Train Final K-Means Model (K=4)

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

# Analyze each cluster
cluster_summary = rfm.groupby('cluster').agg(
    count=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_success_rate=('success_rate', 'mean'),
).round(2)

print('Cluster Summary:')
print(cluster_summary)

# Assign meaningful labels based on cluster characteristics
# Sort by monetary value descending to assign labels
cluster_order = cluster_summary.sort_values('avg_monetary', ascending=False).index.tolist()
label_names = ['VIP', 'Loyal', 'At Risk', 'Lost']

segment_mapping = {}
for i, cluster_id in enumerate(cluster_order):
    label = label_names[i] if i < len(label_names) else f'Segment {i}'
    segment_mapping[cluster_id] = {
        'name': label,
        'description': {
            'VIP': 'High value, frequent, recent buyers',
            'Loyal': 'Regular customers with good purchase history',
            'At Risk': 'Previously active customers showing decline',
            'Lost': 'Inactive customers with no recent purchases',
        }.get(label, ''),
    }

rfm['segment'] = rfm['cluster'].map(lambda c: segment_mapping[c]['name'])

print('\nSegment Distribution:')
print(rfm['segment'].value_counts())

print('\nSegment Mapping:')
for k, v in segment_mapping.items():
    print(f'  Cluster {k} -> {v["name"]}: {v["description"]}')

## 5. Visualize Segments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'VIP': 'gold', 'Loyal': 'green', 'At Risk': 'orange', 'Lost': 'red'}

# Recency vs Monetary
for seg in rfm['segment'].unique():
    subset = rfm[rfm['segment'] == seg]
    axes[0].scatter(subset['recency'], subset['monetary'].clip(upper=100000),
                    alpha=0.3, label=seg, color=colors.get(seg, 'gray'), s=10)
axes[0].set_title('Recency vs Monetary')
axes[0].set_xlabel('Recency (days)')
axes[0].set_ylabel('Monetary (DZD)')
axes[0].legend()

# Segment sizes
rfm['segment'].value_counts().plot(kind='bar', ax=axes[1],
    color=[colors.get(s, 'gray') for s in rfm['segment'].value_counts().index])
axes[1].set_title('Customers per Segment')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Average metrics by segment
seg_means = rfm.groupby('segment')[['recency', 'frequency', 'monetary']].mean()
seg_means.plot(kind='bar', ax=axes[2])
axes[2].set_title('Average RFM by Segment')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
joblib.dump(kmeans, MODEL_DIR / 'segmenter.joblib')
joblib.dump(scaler, MODEL_DIR / 'segmenter_scaler.joblib')
joblib.dump(segment_mapping, MODEL_DIR / 'segment_mapping.joblib')

print(f'Models saved to {MODEL_DIR}')
print(f'  - segmenter.joblib')
print(f'  - segmenter_scaler.joblib')
print(f'  - segment_mapping.joblib')
print('\nDone! The segmentation model is ready for the FastAPI service.')